# 01 - Exploratory data analysis

Shape, class balance, missing values, data-quality problems and default rate by key variables for `cs-training.csv`.
Run from the `notebooks/` folder after downloading the data (see `data/README.md`).

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data import LATE_COLS, TARGET, clean

raw = pd.read_csv("../data/cs-training.csv").drop(columns="Unnamed: 0")
raw.shape

## Class balance and missing values

In [ ]:
print(raw[TARGET].value_counts(normalize=True).round(4))
print()
print((raw.isna().mean() * 100).round(2)[lambda s: s > 0])

## Data-quality problems

In [ ]:
print("Rows with age < 18 or > 100:", (~raw["age"].between(18, 100)).sum())
print("Rows with 96/98 placeholder codes:", (raw[LATE_COLS] >= 96).any(axis=1).sum())
print("Default rate of placeholder rows:", raw.loc[(raw[LATE_COLS] >= 96).any(axis=1), TARGET].mean().round(3))
print("Exact duplicate rows:", raw.duplicated().sum())
raw[["RevolvingUtilizationOfUnsecuredLines", "DebtRatio", "MonthlyIncome"]].describe(percentiles=[.5, .99, .999]).round(2)

## After cleaning

In [ ]:
df = clean(raw)
print(f"Default rate after cleaning: {df[TARGET].mean():.2%}")

## Default rate by key variables

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
bands = {
    "age": ([17, 25, 35, 45, 55, 65, 101], ["<25", "25-34", "35-44", "45-54", "55-64", "65+"]),
    "RevolvingUtilizationOfUnsecuredLines": ([-1, 0.1, 0.3, 0.6, 1.0, 10], ["<10%", "10-30%", "30-60%", "60-100%", "over limit"]),
}
for a, (col, (edges, labels)) in zip(ax, bands.items()):
    rate = df.groupby(pd.cut(df[col], edges, labels=labels), observed=True)[TARGET].mean() * 100
    rate.plot.bar(ax=a, color="#4C78A8", rot=0)
    a.set_title(f"Default rate (%) by {col}")
    a.set_xlabel("")
total_late = df[LATE_COLS].sum(axis=1)
rate = df.groupby(total_late.clip(upper=5))[TARGET].mean() * 100
rate.plot.bar(ax=ax[2], color="#E45756", rot=0)
ax[2].set_title("Default rate (%) by total past-due events (5 = 5+)")
ax[2].set_xlabel("")
plt.tight_layout()